# MedSkel AI demo: U-Net vessel segmentation on FIVES

This notebook loads the trained model and shows what it does, step by step.

- It needs the trained checkpoint `runs/unet_fives/best.pt` and the FIVES data at `C:\data\FIVES`.
- The pictures below are **validation** images (taken from the official training set, never used for weight updates).
- The official test set is **not** re-evaluated here. Section 5 only reads the saved result file.
- Research prototype for a university class. Not a medical tool.

## 1. Load the model

In [ ]:
import csv, json
from pathlib import Path
import numpy as np, torch
import matplotlib.pyplot as plt
from medskel_ai.data import list_pairs, make_split, load_image, load_mask, resize_image_bilinear, resize_mask_nearest
from medskel_ai.model import count_parameters
from medskel_ai.predict import load_model, predict_probability, make_overlay
from medskel_ai.evaluate import confusion_counts, metrics_from_counts

RUN = Path('runs/unet_fives')
model, device, threshold = load_model(RUN / 'best.pt')
print('device:', device)
print(f'trainable parameters: {count_parameters(model):,}')
print('threshold chosen on validation:', threshold)

## 2. Tensor shapes: one 512 x 512 crop through the network

The output has the same height and width as the input, with **one logit per pixel**.

In [ ]:
x = torch.rand(1, 3, 512, 512, device=device)
with torch.no_grad():
    e1 = model.enc1(x); e2 = model.enc2(model.pool(e1)); e3 = model.enc3(model.pool(e2))
    e4 = model.enc4(model.pool(e3)); b = model.bottleneck(model.pool(e4))
    out = model(x)
for name, t in [('input', x), ('enc1', e1), ('enc2', e2), ('enc3', e3), ('enc4', e4), ('bottleneck', b), ('output logits', out)]:
    print(f'{name:14s} {list(t.shape)}')

## 3. Training curves (read from the real training log)

In [ ]:
history = list(csv.DictReader(open(RUN / 'history.csv')))
ep = [int(h['epoch']) for h in history]
fig, (a, b2) = plt.subplots(1, 2, figsize=(11, 3.8))
a.plot(ep, [float(h['train_loss']) for h in history], label='train loss')
a.plot(ep, [float(h['val_loss']) for h in history], label='validation loss')
a.set_xlabel('epoch'); a.set_ylabel('BCE + soft Dice'); a.legend(); a.grid(alpha=.3)
b2.plot(ep, [float(h['val_dice']) for h in history], color='green'); b2.set_xlabel('epoch'); b2.set_ylabel('validation Dice (fixed crops)'); b2.grid(alpha=.3)
plt.show()
best = max(history, key=lambda h: float(h['val_dice']))
print(f"{len(history)} epochs in {float(history[-1]['minutes']):.1f} min; best epoch {best['epoch']} with validation Dice {float(best['val_dice']):.4f}")

## 4. Predict whole validation images

The image is cut into overlapping 512 x 512 tiles, each tile is predicted, and the probabilities are averaged where tiles overlap. One image per disease group (the first validation image of each group; a fixed rule, not hand-picked).

In [ ]:
_, val_pairs = make_split(list_pairs(r'C:\data\FIVES', 'train'), seed=42)
chosen = [next(p for p in val_pairs if p['group'] == g) for g in 'ADGN']
for pair in chosen:
    image, mask = load_image(pair['image']), load_mask(pair['mask'])
    prob = predict_probability(model, image, device)
    pred = prob >= threshold
    m = metrics_from_counts(*confusion_counts(torch.from_numpy(pred), torch.from_numpy(mask).bool()))
    s = 512
    panels = [('image', resize_image_bilinear(image, s, s)), ('expert mask', resize_mask_nearest(mask, s, s) * 255),
              ('prediction', resize_mask_nearest(pred.astype(np.uint8), s, s) * 255), ('overlay', resize_image_bilinear(make_overlay(image, pred), s, s))]
    fig, axes = plt.subplots(1, 4, figsize=(16, 4.3))
    for ax, (title, panel) in zip(axes, panels):
        ax.imshow(panel, cmap='gray', vmin=0, vmax=255); ax.set_title(title); ax.axis('off')
    plt.show()
    print(pair['id'], '(validation)', ' '.join(f'{k}={v:.3f}' for k, v in m.items()))

## 5. Final numbers (saved files, nothing is recomputed here)

In [ ]:
for split in ('val', 'test'):
    f = RUN / f'{split}_metrics.json'
    if not f.exists():
        print(split, ': not evaluated yet'); continue
    r = json.loads(f.read_text())
    print(f"\n{split.upper()}  images={r['images']}  threshold={r['threshold']}")
    print('  overall  ' + '  '.join(f"{k}={v['mean']:.4f}" for k, v in r['per_image'].items()))
    for g, v in r['by_group'].items():
        print(f'  group {g}  ' + '  '.join(f"{k}={v[k]['mean']:.4f}" for k in r['per_image']))
    print('  all-black baseline accuracy:', round(r['all_background_baseline']['accuracy'], 4), '(Dice 0, recall 0)')